<div align="center">

# Week 14: Human-AI Interaction II: Conversational Interfaces, Dark Patterns and Ethics

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-14) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/Week14_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the values of the three slots of a booking: The departments, the days and the times. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The values of the three slots of a booking
DEPTS = ["Cardiology", "Chest diseases", "Gastroenterology", "Neurology",
         "Orthopedics", "Dermatology"]
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
TIMES = ['09:00', '10:30', '13:00', '14:30', '16:00']
SLOTS = {"department": DEPTS, "day": DAYS, "time": TIMES}
print("Setup complete. Running in Colab:", IN_COLAB, "| slots:", list(SLOTS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 14.1": (lambda: 13.5, 1e-6),
    "Exercise 14.2": (lambda: 'booked', 0),
    "Exercise 14.3": (lambda: 5, 0),
    "Exercise A": (lambda: 'handover', 0),
    "Exercise B": (lambda: 0.9 ** 4, 1e-6),
    "Exercise C": (lambda: 0.8 * 1.0 - 0.45 * (-0.5), 1e-6),
    "Exercise D": (lambda: 7 * (1.35 + 1.10 + 0.20), 1e-6),
    "Exercise E": (lambda: 3 / 8, 1e-6),
    "Exercise F": (lambda: 11 / 24, 1e-6),
}

## Python step 14: Organised and tested code

A booking assistant runs many small pieces of code: One understands a value, one decides what to ask next, and one books. Each piece must do what it promises, also after the next change. This step writes functions that explain themselves and tests them with `assert` &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#ref-1)&#93;. It then stores the behaviour of a dialogue as a table of states and events, and it checks many cases at once with a small test runner. The sections of the week use these pieces for the dialogue manager, its tests and its evaluation.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "SLOTS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Functions that explain themselves

In [ ]:
# Python practice: a small example for learning the language
def minutes_between(start, end):
    """Minutes from start to end, both written as "HH:MM"."""
    h1, m1 = map(int, start.split(":"))
    h2, m2 = map(int, end.split(":"))
    return (h2 * 60 + m2) - (h1 * 60 + m1)

print(minutes_between("09:00", "10:30"))
help(minutes_between)

The text in triple quotes under `def` is the *docstring*: It says what the function does, and the function `help` shows it. The method `split(":")` cuts the text at the colon. The function `map(int, ...)` then turns both parts into whole numbers. A function with a clear name, a docstring and one job is easier to test and to change.

### Tests with assert

In [ ]:
# Python practice: a small example for learning the language
def test_minutes_between():
    assert minutes_between("09:00", "10:30") == 90
    assert minutes_between("13:00", "13:00") == 0
    assert minutes_between("16:00", "14:30") == -90

test_minutes_between()
print("all tests passed")
try:
    assert minutes_between("09:00", "10:30") == 60, "90 minutes expected"
except AssertionError as err:
    print("a failing test says:", err)

`assert` checks a condition and stops with an `AssertionError` when it is false. A test function collects such checks for one function, including the edge cases: The same time, and an end before the start. `try` and `except` catch the error. The program can then report it instead of stopping.

### A table of states and events

In [ ]:
# Python practice: a small example for learning the language
TABLE = {("start", "hello"): "ask_department",
         ("ask_department", "value"): "ask_day",
         ("ask_day", "value"): "ask_time",
         ("ask_time", "value"): "confirm",
         ("confirm", "yes"): "booked",
         ("confirm", "no"): "ask_department"}

def next_state(state, event):
    """The next state; an event that the table does not know keeps the state."""
    return TABLE.get((state, event), state)

state = "start"
for event in ["hello", "value", "value", "cough", "value", "yes"]:
    state = next_state(state, event)
    print(f"{event:>6} -> {state}")

A dictionary whose keys are pairs maps a state and an event to the next state. `TABLE.get(key, state)` returns the state itself when the pair is missing. An unknown event such as `cough` then does not break the dialogue. `{event:>6}` pads the event to six characters on the left.

### Many cases at once: A small test runner

In [ ]:
# Python practice: a small example for learning the language
def final_state(events, state="start"):
    for e in events:
        state = next_state(state, e)
    return state

CASES = [(["hello", "value", "value", "value", "yes"], "booked"),
         (["hello", "value", "value", "value", "no"], "ask_department"),
         (["hello", "yes"], "ask_department"),
         ([], "start")]
passed = 0
for events, expected in CASES:
    got = final_state(events)
    passed += got == expected
    print("ok  " if got == expected else "FAIL", events, "->", got)
print(f"{passed} of {len(CASES)} cases passed")

Each case pairs a list of events with the state expected at the end. The loop runs every case and counts the passes, since `True` adds as 1. A table of cases grows with every error found, so the same error cannot return unnoticed.

**Quick check.** Why does `next_state` keep the state for an unknown event instead of raising an error?

<details><summary>Answer</summary>

A patient may say anything. Keeping the state lets the assistant ask again instead of crashing; a test case with an unknown event checks this behaviour.

</details>

### Exercises for Python step 14

**Exercise 14.1.** Store the hours from 08:15 to 21:45 in `hours`, with `minutes_between`.

In [ ]:
hours = None   # your answer

check("Exercise 14.1", hours)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
hours = minutes_between("08:15", "21:45") / 60

check("Exercise 14.1", hours)

**Exercise 14.2.** Store the final state of the events hello, value, value, value, no, value, value, value and yes in `end`.

In [ ]:
end = None   # your answer

check("Exercise 14.2", end)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
end = final_state(["hello"] + ["value"] * 3 + ["no"] + ["value"] * 3 + ["yes"])

check("Exercise 14.2", end)

**Exercise 14.3.** Store the number of events of the longest case in `CASES` in `n_long`.

In [ ]:
n_long = None   # your answer

check("Exercise 14.3", n_long)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_long = max(len(events) for events, _ in CASES)

check("Exercise 14.3", n_long)

## 1. A dialogue manager as a state machine &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#ref-2)&#93;

*Lecture: [Conversational interfaces: Turns, intents and slots](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#conversational-interfaces)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "final_state" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The booking assistant of the clinic asks for three slots: The department, the day and the time. Python writes its dialogue manager as a finite-state machine, tests it and simulates conversations in which the recognizer sometimes mishears a value. It then analyses your bookings of Part A.

**From the formulas to the code.** `FSM` maps a state and an event to the next state, as in Python step 14. `converse(target, strategy, e, theta, rng)` simulates one booking. A value is misheard with the probability $e$, and the recognizer reports a confidence $c$. With the strategy `confidence`, the assistant asks back when $c < \theta$.

In [ ]:
FSM = {("start", "hello"): "ask_department",
       ("ask_department", "value"): "ask_day", ("ask_day", "value"): "ask_time",
       ("ask_time", "value"): "confirm", ("confirm", "yes"): "booked",
       ("confirm", "no"): "ask_department"}
ANY = ["person"]                      # events that work in every state

def step(state, event):
    """Next state of the dialogue; asking for a person always hands over."""
    if event in ANY:
        return "handover"
    return FSM.get((state, event), state)

def test_fsm():
    assert step("start", "hello") == "ask_department"
    assert step("ask_time", "value") == "confirm"
    assert step("confirm", "no") == "ask_department"
    assert step("ask_day", "person") == "handover"
    assert step("ask_day", "cough") == "ask_day"

test_fsm()
print("the dialogue manager passes its tests")
state = "start"
for ev in ["hello", "value", "value", "value", "yes"]:
    state = step(state, ev)
print("a booking without errors ends in:", state)

**Python note.** `test_fsm` is a test function as in Python step 14. It covers every kind of transition, the handover from any state and an event that the table does not know.

**Exercise A.** A patient says hello, gives the department and then asks for a person. Store the state after these three events in `st`, with `step`.

In [ ]:
st = None   # your answer

check("Exercise A", st)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
st = "start"
for ev in ["hello", "value", "person"]:
    st = step(st, ev)

check("Exercise A", st)

In [ ]:
E, THETA = 0.1, 0.8          # error rate of the recognizer, threshold
RIGHT_C, WRONG_C = (0.7, 1.0), (0.3, 0.9)   # ranges of the confidence

def hear(value, slot, e, rng):
    """What the recognizer hears, and its confidence."""
    if rng.random() < e:
        other = rng.choice([v for v in SLOTS[slot] if v != value])
        return other, rng.uniform(*WRONG_C)
    return value, rng.uniform(*RIGHT_C)

def converse(target, strategy="none", e=E, theta=THETA, rng=random):
    """Turns of the patient and whether the booking is right."""
    turns, booked = 0, []
    for slot, value in zip(SLOTS, target):
        while True:
            heard, c = hear(value, slot, e, rng)
            turns += 1
            ask = strategy == "explicit" or (strategy == "confidence"
                                              and c < theta)
            if not ask:
                break
            turns += 1                # yes or no
            if heard == value:
                break
        booked.append(heard)
    return turns, tuple(booked) == tuple(target)

rng = random.Random(RANDOM_STATE)
target = ("Dermatology", "Thursday", "10:30")
for strategy in ("none", "explicit", "confidence"):
    runs = [converse(target, strategy, rng=rng) for _ in range(20_000)]
    print(f"{strategy:>10}: {np.mean([t for t, _ in runs]):.3f} turns, "
          f"{np.mean([ok for _, ok in runs]):.3f} right bookings")

**Python note.** `zip(SLOTS, target)` pairs the slot names with the values of the booking. `while True` repeats until `break`: With asking back, a misheard value is refused and asked again. `rng.uniform(*WRONG_C)` unpacks the tuple into the two bounds of the range.

**Paste your results.** Copy the results of Part A of the interactive lab and paste them between the triple quotes of `MY_CHAT` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_CHAT = """
"""
DEFAULT_CHAT = """
round,booking,strategy,turns,wrong_slots,right,ms
1,1,none,3,0,1,14785
1,2,none,3,1,0,11403
1,3,none,3,0,1,13172
1,4,none,3,1,0,14766
2,1,confidence,5,0,1,18158
2,2,confidence,4,0,1,19057
2,3,confidence,5,0,1,19046
2,4,confidence,4,0,1,17546"""
src = MY_CHAT.strip() or DEFAULT_CHAT.strip()
print("Data:", "your own bookings from the interactive lab" if MY_CHAT.strip()
      else "the results of a typical student")
chat = pd.read_csv(io.StringIO(src))
for rd, part in chat.groupby("round"):
    print(f"Round {rd} ({part.strategy.iloc[0]}): {part.turns.sum()} turns, "
          f"{part.right.sum()} of {len(part)} bookings right, mean time "
          f"{part.ms.mean() / 1000:.1f} s per booking")

**Python note.** `part.strategy.iloc[0]` takes the strategy from the first row of the round. Every booking of a round uses the same strategy.

**Observe and reflect.** In round 1, did you notice the misheard values before the booking was made? What would have let you notice them?

## 2. Errors, confirmations and the length of a dialogue

*Lecture: [Errors and confirmations](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#confirmation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "converse" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Asking back catches misheard values, but every question costs a turn. Python computes the expected turns and the share of right bookings for three strategies. It checks the formulas against the simulation of Section 1 and draws the trade-off for several error rates.

**From the formulas to the code.** Without asking back, `none_` gives $P = (1 - e)^k$ right bookings with $T = k$ turns. Asking back every value, `explicit_` gives $T = \frac{2k}{1 - e}$ and $P = 1$. Asking back below $\theta$, `confidence_` gives per slot $T_1 = \frac{r(2 - a) + e(2 - b)}{1 - e(1 - b)}$ and $P_1 = \frac{r}{r + e\,b}$, with $r = 1 - e$. $a$ and $b$ are the shares of right and of wrong values whose confidence reaches $\theta$.

In [ ]:
K = 3                                  # slots of a booking

def shares(theta=THETA):
    """Shares of right (a) and wrong (b) values with a confidence >= theta."""
    a = (RIGHT_C[1] - max(theta, RIGHT_C[0])) / (RIGHT_C[1] - RIGHT_C[0])
    lo = min(max(theta, WRONG_C[0]), WRONG_C[1])
    b = (WRONG_C[1] - lo) / (WRONG_C[1] - WRONG_C[0])
    return round(a, 3), round(b, 3)

def none_(e=E, k=K):
    return k, (1 - e) ** k

def explicit_(e=E, k=K):
    return 2 * k / (1 - e), 1.0

def confidence_(e=E, k=K, theta=THETA):
    a, b = shares(theta)
    r = 1 - e
    t1 = (r * (2 - a) + e * (2 - b)) / (1 - e * (1 - b))
    p1 = r / (r + e * b)
    return k * t1, p1 ** k

print("a, b at theta 0.8:", shares())
for f in (none_, explicit_, confidence_):
    t, p = f()
    print(f"{f.__name__:>12}: {t:.3f} turns, {p:.3f} right bookings")

**Python note.** The functions end with an underscore, because `none` would be easy to confuse with Python's `None`. `f.__name__` is the name of a function, and the loop runs the three functions like any other values.

**Exercise B.** A booking of four slots, without asking back, with $e = 0.1$: Store the share of right bookings in `p4`.

In [ ]:
p4 = None   # your answer

check("Exercise B", p4)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p4 = none_(0.1, 4)[1]

check("Exercise B", p4)

In [ ]:
# Figure: turns and right bookings of the three strategies for several error rates
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
es = np.linspace(0.0, 0.3, 31)
for f, col in ((none_, "#B53F2E"), (explicit_, "#5A4BB5"), (confidence_, "#2E8B57")):
    vals = np.array([f(e) for e in es])
    ax[0].plot(es, vals[:, 0], color=col, label=f.__name__.strip("_"))
    ax[1].plot(es, vals[:, 1], color=col, label=f.__name__.strip("_"))
for e in (0.05, 0.1, 0.2):
    sims = [converse(target, "confidence", e=e, rng=rng) for _ in range(5000)]
    t, p = np.mean(sims, axis=0)
    ax[0].plot(e, t, "o", color="#2E8B57")
    ax[1].plot(e, p, "o", color="#2E8B57")
ax[0].set_xlabel("error rate of the recognizer e")
ax[0].set_ylabel("turns of the patient")
ax[1].set_xlabel("error rate of the recognizer e")
ax[1].set_ylabel("share of right bookings")
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The dots are simulations of Section 1 with the strategy `confidence`, and they lie on the lines of the formulas. `np.mean(..., axis=0)` averages the turns and the right bookings separately.

**Explore.** Move the sliders: The code prints the turns and the right bookings of the three strategies. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(e=0.1, theta=0.8):
    for f in (none_, explicit_):
        t, p = f(e)
        print(f"{f.__name__:>12}: {t:.3f} turns, {p:.3f} right")
    t, p = confidence_(e, theta=theta)
    print(f"{'confidence_':>12}: {t:.3f} turns, {p:.3f} right")

widgets.interact(explore,
                 e=widgets.FloatSlider(value=0.1, min=0.0, max=0.4, step=0.01),
                 theta=widgets.FloatSlider(value=0.8, min=0.3, max=1.0, step=0.05));

**Observe and reflect.** At which error rate would you switch the clinic from asking back below the threshold to asking back every value?

## 3. Evaluating conversations by success and costs &#91;[3](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#ref-3)&#93;

*Lecture: [Evaluating conversations](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#evaluating-conversations)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "confidence_" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Which strategy do patients prefer? PARADISE (PARAdigm for DIalogue System Evaluation) explains user satisfaction by task success and by the costs of a dialogue. Python simulates 90 rated bookings and fits the weights of success and turns by linear regression. It then predicts the satisfaction of each strategy.

**From the formulas to the code.** `z(x)` computes $\mathcal{N}(x) = \frac{x - \bar x}{s_x}$. The model of user satisfaction (US) is $\mathrm{US} = \beta_0 + \alpha\,\mathcal{N}(\text{success}) - w\,\mathcal{N}(\text{turns})$, and `np.linalg.lstsq` finds $\beta_0$, $\alpha$ and $w$.

In [ ]:
def z(x):
    """Normalise to mean 0 and standard deviation 1."""
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std()

rng = random.Random(RANDOM_STATE)
rows = []
for strategy in ("none", "explicit", "confidence"):
    for _ in range(30):
        tgt = (rng.choice(DEPTS), rng.choice(DAYS), rng.choice(TIMES))
        turns, ok = converse(tgt, strategy, rng=rng)
        rows.append((strategy, turns, int(ok)))
logs = pd.DataFrame(rows, columns=["strategy", "turns", "success"])
# the ratings of the simulated patients, from 1 to 5 (illustrative)
noise = np.random.default_rng(RANDOM_STATE).normal(0, 0.35, len(logs))
logs["US"] = np.clip(3.5 + 0.8 * z(logs.success) - 0.45 * z(logs.turns) + noise,
                     1, 5).round(1)
X = np.column_stack([np.ones(len(logs)), z(logs.success), z(logs.turns)])
beta, *_ = np.linalg.lstsq(X, logs.US, rcond=None)
print("fitted: US = {:.3f} + {:.3f} N(success) - {:.3f} N(turns)".format(
    beta[0], beta[1], -beta[2]))
logs["predicted"] = X @ beta
print(logs.groupby("strategy")[["turns", "success", "US", "predicted"]]
      .mean().round(3))

**Python note.** `np.linalg.lstsq` solves the least squares of Week 4 for several predictors at once; `beta, *_ = ...` keeps the first result and drops the rest. The column of ones carries the constant $\beta_0$.

**Exercise C.** A booking is 1.0 standard deviations above the mean in success and 0.5 below it in turns. With $\alpha = 0.8$ and $w = 0.45$, store $\alpha\,\mathcal{N}(\text{success}) - w\,\mathcal{N}(\text{turns})$ in `perf`.

In [ ]:
perf = None   # your answer

check("Exercise C", perf)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
perf = 0.8 * 1.0 - 0.45 * (-0.5)

check("Exercise C", perf)

In [ ]:
# Figure: ratings against turns, by strategy and success
fig, ax = plt.subplots(figsize=(7, 3.6))
cols = {"none": "#B53F2E", "explicit": "#5A4BB5", "confidence": "#2E8B57"}
jit = np.random.default_rng(1).uniform(-0.15, 0.15, len(logs))
for s, part in logs.groupby("strategy"):
    ax.scatter(part.turns + jit[part.index], part.US, s=18, color=cols[s],
               marker="o", label=s, alpha=0.7)
ax.set_xlabel("turns of the patient")
ax.set_ylabel("user satisfaction (1 to 5)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** A small random shift, the *jitter*, spreads the points that share the same number of turns, so that they do not hide each other.

**Observe and reflect.** PARADISE weighs success and costs by what users report. What could a clinic miss if it asked only patients who finished their booking?

## 4. Dark patterns, consent and cancellation &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#ref-4)&#93;

*Lecture: [Dark patterns](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#dark-patterns)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "beta" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A dark pattern makes one choice easy and the other hard. Python predicts the time of the paths of Part C with the keystroke-level model of Week 6. It estimates how many patients would consent to marketing when the refusal is hidden, and it analyses your results of Part C.

**From the formulas to the code.** `path_time(taps)` computes $T = n\,(t_M + t_P + t_B)$ for $n$ taps, each with a decision. The consent model adds the effect $\Delta$ measured by Nouwens and colleagues to the consent $p_0$ of a fair screen: $p = p_0 + \Delta$.

In [ ]:
KLM = {'P': 1.1, 'B': 0.2, 'M': 1.35}    # seconds, Week 6
TAP = round(KLM["M"] + KLM["P"] + KLM["B"], 2)       # a tap with a decision

def path_time(taps):
    return round(taps * TAP, 2)

PATHS = {   # taps of each path of Part C
    ('dark', 'consent', 'refuse'): 4,
    ('dark', 'consent', 'accept'): 1,
    ('fair', 'consent', 'refuse'): 1,
    ('fair', 'consent', 'accept'): 1,
    ('dark', 'cancel', 'cancel'): 7,
    ('fair', 'cancel', 'cancel'): 2}
for (design, task, goal), taps in PATHS.items():
    word = "tap" if taps == 1 else "taps"
    print(f"{design:>4} {task:>7} ({goal}): {taps} {word}, {path_time(taps)} s")
P_FAIR = 0.2    # consent to marketing with a fair screen (illustrative)
DELTA = 0.225    # more consent when the refusal is hidden
for n in (1000,):
    print(f"of {n} patients: {round(n * P_FAIR)} consent with a fair screen, "
          f"{round(n * (P_FAIR + DELTA))} when the refusal is hidden")

**Python note.** A dictionary with tuples as keys unpacks into three names in the `for` line. The fair rate of 0.20 is illustrative; the difference of 0.225 is the middle of the 22 to 23 points that Nouwens and colleagues measured on the web &#91;[8](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#ref-8)&#93;.

**Exercise D.** Store the predicted time of the dark cancellation path, seven taps with a decision, in `t_dark`.

In [ ]:
t_dark = None   # your answer

check("Exercise D", t_dark)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_dark = path_time(7)

check("Exercise D", t_dark)

**Paste your results.** Copy the results of Part C of the interactive lab and paste them between the triple quotes of `MY_CONSENT` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_CONSENT = """
"""
DEFAULT_CONSENT = """
round,design,task,taps,ms,outcome
1,dark,consent,1,6197,1
1,dark,cancel,9,39941,1
2,fair,consent,1,4055,0
2,fair,cancel,2,6210,1"""
src = MY_CONSENT.strip() or DEFAULT_CONSENT.strip()
print("Data:", "your own tasks from the interactive lab" if MY_CONSENT.strip()
      else "the results of a typical student")
dp = pd.read_csv(io.StringIO(src))
for _, r in dp.iterrows():
    goal = ("consented to marketing" if r.outcome else "refused marketing") \
        if r.task == "consent" else ("cancelled" if r.outcome else "not cancelled")
    word = "tap" if r.taps == 1 else "taps"
    print(f"{r.design:>4} {r.task:>7}: {r.taps} {word} in {r.ms / 1000:.1f} s, "
          f"{goal} (KLM: {path_time(r.taps)} s)")

**Python note.** `iterrows()` walks through the rows of a table, one at a time. In the consent task the goal was to refuse marketing, so an outcome of 1 there means that the design won.

**Observe and reflect.** Which part of the dark designs of Part C slowed you down most: The hidden refusal, the questions that shamed you or the number of screens?

## 5. Handing over to a person &#91;[9](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#ref-9)&#93;

*Lecture: [Handing over to a person](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#handover)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "path_time" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A chatbot must know when a person should take over. Python applies a rule to the 24 conversations of Part D: Take over when an urgent word appears or the confidence of the chatbot is low. It computes how many conversations that need a person the rule finds and how many the supervisor must read.

**From the formulas to the code.** `rule(text, c, theta)` takes over when a word of `URGENT` appears or $c < \theta$. The rule has true positives (TP), false negatives (FN) and false positives (FP). The recall $\frac{TP}{TP + FN}$ is the share of the conversations that need a person and are taken over. The precision $\frac{TP}{TP + FP}$ is the share of the taken over that need a person.

In [ ]:
CONVS = [   # round, patient, chatbot, confidence, needs a person, reason
    (1, 'When does the laboratory open on Saturday?',
     'The laboratory is open on Saturdays from 8:00 to 12:00.',
     0.95, 0, ''),
    (1, 'I need to move my dermatology appointment to next week.',
     'I can move it. Is Tuesday at 13:00 next week fine?',
     0.91, 0, ''),
    (1, 'Can my daughter come with me to the appointment?',
     'Yes, one companion can come with you.',
     0.87, 0, ''),
    (1, 'Do I need to fast before my blood test?',
     'For most blood tests, yes. Your letter says which tests you have.',
     0.84, 0, ''),
    (1, "That's not what I said. I want to cancel, not change!",
     'I can help you change your appointment. Which day suits you?',
     0.42, 1, 'the chatbot failed twice'),
    (1, 'Where can I park my car?',
     'The visitor car park is behind building B. The first hour is free.',
     0.93, 0, ''),
    (1, 'Please delete all my data from your app.',
     'You can change your notification settings in the menu.',
     0.48, 1, 'a request about personal data'),
    (1, 'Is there a pharmacy in the hospital?',
     'Yes, on the ground floor next to the main entrance.',
     0.96, 0, ''),
    (1, 'I will be ten minutes late for my 14:30 appointment.',
     'Thank you for telling us. Please check in at the kiosk when you '
     'arrive.',
     0.83, 0, ''),
    (1, 'Can I talk to a real person, please?',
     'I can answer most questions. What would you like to know?',
     0.55, 1, 'asks for a person'),
    (1, 'I have crushing pain in my chest and my left arm feels numb.',
     'I can book Cardiology for you. The next free time is Thursday '
     'at 10:30.',
     0.81, 1, 'urgent symptoms'),
    (1, 'Are the results of my chest X-ray ready?',
     'Your results are in the app under Results once the doctor has '
     'seen them.',
     0.88, 0, ''),
    (2, 'Can you send my appointment to my e-mail?',
     'Yes. I have sent it to the address in your profile.',
     0.9, 0, ''),
    (2, 'Book me a check-up at Cardiology next month.',
     'The first free check-up next month is on Monday the 7th at '
     '09:00. Shall I book it?',
     0.94, 0, ''),
    (2, 'I want to file a complaint about a nurse who was rude to me.',
     'Thank you for your feedback! Would you like to rate your visit?',
     0.51, 1, 'a complaint'),
    (2, 'I took twice my dose of blood thinners this morning by mistake.',
     'Your next appointment at Cardiology is on Friday at 09:00.',
     0.71, 1, 'urgent symptoms'),
    (2, 'Can I get a copy of my chest X-ray for my family doctor?',
     'Yes. You can download it in the app under Results.',
     0.86, 0, ''),
    (2, 'You booked the wrong department twice now. This is useless.',
     'I am sorry. Which department would you like?',
     0.47, 1, 'the chatbot failed twice'),
    (2, 'Can I see another doctor at my next appointment?',
     'I can ask the department. Which doctor would you prefer?',
     0.85, 0, ''),
    (2, 'Is the cafeteria open on Sundays?',
     'Yes, from 9:00 to 17:00.',
     0.95, 0, ''),
    (2, 'Which documents do I need for my first visit?',
     'Your identity card and, if you have one, the referral letter.',
     0.92, 0, ''),
    (2, 'My gums are bleeding after a new toothpaste. Which clinic '
     'should I see?',
     'Gum problems are seen by the dental clinic. I can book it for you.',
     0.77, 0, ''),
    (2, 'My father fell and he is not responding to me.',
     'I can book Orthopedics for your father. Which day suits him?',
     0.62, 1, 'urgent symptoms'),
    (2, 'How much does a parking ticket cost?',
     'The first hour is free, and every further hour costs 20 lira.',
     0.93, 0, '')]
URGENT = ["chest", "numb", "not responding", "breathe", "bleeding", "dose",
          "fell", "unconscious"]

def flagged(text):
    return any(w in text.lower() for w in URGENT)

def rule(text, c, theta=0.6):
    return flagged(text) or c < theta

def evaluate(theta=0.6, use_flag=True):
    tp = fp = fn = 0
    for rd, pat, bot, c, need, why in CONVS:
        take = (flagged(pat) and use_flag) or c < theta
        tp += take and need
        fp += take and not need
        fn += need and not take
    return tp / (tp + fn), tp / max(tp + fp, 1), (tp + fp) / len(CONVS)

for theta in (0.0, 0.5, 0.6, 0.7, 0.8):
    rec, prec, read = evaluate(theta)
    print(f"theta {theta}: recall {rec:.3f}, precision {prec:.3f}, "
          f"read {read:.3f} of the conversations")

**Python note.** With `theta=0.0`, no confidence is low enough, so only the urgent words decide. `max(tp + fp, 1)` avoids a division by zero when the rule takes over nothing.

**Exercise E.** Store the recall of the urgent words alone, without the confidence, in `rec_words`.

In [ ]:
rec_words = None   # your answer

check("Exercise E", rec_words)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rec_words = evaluate(0.0)[0]

check("Exercise E", rec_words)

**Paste your results.** Copy the results of Part D of the interactive lab and paste them between the triple quotes of `MY_NIGHT` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_NIGHT = """
"""
DEFAULT_NIGHT = """
round,conversation,needs,flag,conf,choice,right,ms
1,1,0,0,0.95,continue,1,6308
1,2,0,0,0.91,continue,1,6935
1,3,0,0,0.87,continue,1,5890
1,4,0,0,0.84,continue,1,7131
1,5,1,0,0.42,take over,1,6580
1,6,0,0,0.93,continue,1,7881
1,7,1,0,0.48,continue,0,4623
1,8,0,0,0.96,take over,0,5364
1,9,0,0,0.83,continue,1,5903
1,10,1,0,0.55,take over,1,6637
1,11,1,1,0.81,take over,1,7425
1,12,0,1,0.88,continue,1,6968
2,1,0,0,0.9,continue,1,5507
2,2,0,0,0.94,continue,1,6640
2,3,1,0,0.51,take over,1,4137
2,4,1,1,0.71,take over,1,6043
2,5,0,1,0.86,take over,0,6925
2,6,1,0,0.47,take over,1,5950
2,7,0,0,0.85,continue,1,4852
2,8,0,0,0.95,continue,1,4885
2,9,0,0,0.92,continue,1,6731
2,10,0,1,0.77,take over,0,4952
2,11,1,1,0.62,take over,1,5384
2,12,0,0,0.93,continue,1,5696"""
src = MY_NIGHT.strip() or DEFAULT_NIGHT.strip()
print("Data:", "your own decisions from the interactive lab" if MY_NIGHT.strip()
      else "the results of a typical student")
nv = pd.read_csv(io.StringIO(src))
for rd, part in nv.groupby("round"):
    take = part.choice == "take over"
    missed = ((part.needs == 1) & ~take).sum()
    extra = ((part.needs == 0) & take).sum()
    print(f"Round {rd}: right {part.right.sum()} of {len(part)} | missed "
          f"{missed} that needed a person | {extra} taken over without need | "
          f"mean time {part.ms.mean() / 1000:.1f} s")

**Python note.** `~take` negates a column of `True` and `False`. A missed conversation needed a person but was left to the chatbot.

In [ ]:
# Figure: recall and reading of the rule against the threshold
fig, ax = plt.subplots(figsize=(7, 3.4))
ths = np.linspace(0, 1, 101)
for use, ls in ((True, "-"), (False, "--")):
    v = np.array([evaluate(t, use) for t in ths])
    lab = "words and confidence" if use else "confidence alone"
    ax.plot(ths, v[:, 0], ls, color="#2E8B57", label=f"recall, https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lab.html")
    ax.plot(ths, v[:, 2], ls, color="#C98510", label=f"read, https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lab.html")
ax.set_xlabel("threshold of the confidence")
ax.set_ylabel("share")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

**Python note.** Solid lines combine the urgent words with the confidence, dashed lines use the confidence alone. The words find the urgent conversations that the chatbot misunderstood with confidence.

**Observe and reflect.** Which conversation of Part D was the hardest to judge, and what would have helped you?

## 6. The clinic app, stage 14

*Lecture: [The clinic app, stage 14: A conversational booking assistant and a review of dark patterns](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#the-clinic-app-stage-14)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "evaluate" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 14 completes the clinic app. The booking assistant asks back below a threshold of confidence, the consent and the cancellation are fair, and a person takes over when the rule calls for it. Python collects the results of the week in one dictionary.

In [ ]:
STAGE14 = {}
for f in (none_, explicit_, confidence_):
    t, p = f()
    name = f.__name__.strip("_")
    STAGE14[f"turns per booking, {name}"] = round(t, 3)
    STAGE14[f"right bookings, {name}"] = round(p, 3)
STAGE14["marketing consent, fair screen"] = P_FAIR
STAGE14["marketing consent, refusal hidden"] = round(P_FAIR + DELTA, 3)
for design in ("dark", "fair"):
    taps = PATHS[(design, "cancel", "cancel")]
    STAGE14[f"cancellation, {design} path (s)"] = path_time(taps)
rec, prec, read = evaluate(0.6)
STAGE14["handover rule, recall"] = round(rec, 3)
STAGE14["handover rule, precision"] = round(prec, 3)
STAGE14["handover rule, conversations read"] = round(read, 3)
for key, value in STAGE14.items():
    print(f"{key}: {value}")

**Python note.** The values come from the functions of Sections 2, 4 and 5. `f.__name__.strip("_")` removes the underscore at the end of each name.

**Exercise F.** Store the share of the 24 conversations that the supervisor reads with the rule at the threshold 0.7 in `read7`.

In [ ]:
read7 = None   # your answer

check("Exercise F", read7)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
read7 = evaluate(0.7)[2]

check("Exercise F", read7)

In [ ]:
# Figure: the booking assistant, consent and cancellation, and the handover
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
names = ["none", "explicit", "confidence"]
ax[0].bar(names, [STAGE14[f"turns per booking, {n}"] for n in names],
          color=["#B53F2E", "#5A4BB5", "#2E8B57"])
ax[0].set_ylabel("turns per booking")
ax0 = ax[0].twinx()
ax0.plot(names, [STAGE14[f"right bookings, {n}"] for n in names], "ko-")
ax0.set_ylim(0, 1.05)
ax0.set_ylabel("right bookings")
ax[1].bar(["fair", "refusal hidden"], [STAGE14["marketing consent, fair screen"],
                                       STAGE14["marketing consent, refusal hidden"]],
          color=["#2E8B57", "#B53F2E"])
ax[1].set_ylabel("consent to marketing")
hand = [STAGE14[f"handover rule, {k}"]
        for k in ("recall", "precision", "conversations read")]
ax[2].bar(["recall", "precision", "read"], hand,
          color=["#2E8B57", "#5A4BB5", "#C98510"])
ax[2].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

**Python note.** `twinx()` adds a second vertical axis on the right of the first panel, so that turns and right bookings share one picture.

## 7. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE14" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-14/"
               "NB14_conversation_dark_patterns_ethics.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The models of this week apply to any conversational service. Each application below has slots, an error rate of its recognizer and a share of the conversations that need a person.

The values are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "airline_rebooking"
# clinic, pizza_order, bank_hotline, airline_rebooking, smart_home
APPS = {   # slots, error rate of the recognizer, share that needs a person
    "clinic": (3, 0.1, 0.33),
    "pizza_order": (4, 0.06, 0.05),
    "bank_hotline": (3, 0.08, 0.3),
    "airline_rebooking": (5, 0.12, 0.25),
    "smart_home": (2, 0.15, 0.02)}

k, e, need = APPS[APP]
for f in (none_, explicit_):
    t, p = f(e, k)
    print(f"{APP}, {f.__name__.strip('_')}: {t:.2f} turns, {p:.3f} right")
t, p = confidence_(e, k)
print(f"{APP}, confidence: {t:.2f} turns, {p:.3f} right")
print(f"  a supervisor reads at least {need:.0%} of the conversations")

**Python note.** `{need:.0%}` prints a share as a percentage without decimals. The functions of Section 2 take the number of slots `k` of the application.

**Observe and reflect.** Which strategy would you choose for your application, and which errors would its users notice first?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Add a state for changing a booking to `FSM`, write three test cases for it and run them with the test runner of Python step 14. |
| Web development | Design the consent screen of a web shop with two buttons of the same size. Count its taps and predict its time with `path_time`. |
| Mobile development | A voice assistant on a phone has an error rate of 0.15. Find the threshold at which `confidence_` reaches 0.95 right bookings with the fewest turns. |
| Game development | A character in a game asks the player three questions. Use `converse` to compare the turns of asking back always and never at an error rate of 0.2. |
| Embedded systems and the Internet of Things | A smart speaker hears commands of two slots with an error rate of 0.1. Compute the turns and the right commands of the three strategies. |
| Data science | Simulate 300 instead of 90 rated bookings, fit PARADISE again and compare the weights with those of this week. |
| Cybersecurity | A phishing chatbot pretends to be the clinic. List three questions with which a patient can check whom it is talking to, and the answers that a fair assistant gives. |
| Health informatics | Add three urgent words to `URGENT` and compute the recall and the precision of the rule on the 24 conversations again. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-14).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>